## 1. Import libraries and configure the inference environment

In [1]:
from    typing      import  Literal, Sequence
from    itertools   import  product

from    tqdm.notebook   import  tqdm
from    time            import  perf_counter

import  torch
from    torch.utils.data                import  DataLoader

from    core.numerical.solver import  FastSM_Boltzmann_VHS, LAMBDA, one_step_RK4_classic
from    core.numerical.math   import  area_of_unit_sphere

from    loader      import  NormKeys, HistKeys
from    loader      import  path_base
from    plot        import  *


DEVICE: torch.device = torch.device("cuda:1")

list_max_freq   = [6, 8, 10]
list_seed       = [0, 1, 2]
max_freq        = list_max_freq[1]
seed            = list_seed[0]

res_v = 2**6

In [2]:
DIMENSION   = 2
MAX_V       = 3/LAMBDA
VHS_COEFF   = 1.0/area_of_unit_sphere(DIMENSION)
min_t       = 0.0
max_t       = 5.0
delta_t     = 0.1

## 2. Instantiate the trained operators and normalizers

## 3. Conduct inference

### 3-1. Construct dataloader

In [3]:
path_data = path_base / "datasets" / "test" / f"Boltzmann_2D__coeff1.59e-01__res_t051_v{res_v:03d}"
assert path_data.exists(), f"The data path {path_data} does not exist."
list_alpha = [0.50]
part_start, part_end = 1, 1

data, params = [], []
generation_time = 0.0
files = sorted(path_data.glob("alpha*.pth"))
alpha_and_part = list(product(list_alpha, range(part_start, part_end+1)))
for alpha, part in tqdm(alpha_and_part):
    if res_v>2**6:
        file = path_data / f"alpha{alpha:.2f}__part{part:03d}.pth"
    else:
        file = path_data / f"alpha{alpha:.2f}__part{part:02d}.pth"
    f = torch.load(file, weights_only=False)
    data.append(f['data'][[0]])         # (num_instances, res_t, 1, 1, res, res, 1)
    params.append(f['vhs_alpha']*torch.ones(1, 1))     # float
    generation_time += f['elapsed_time']
data    = torch.cat(data, dim=0).to(DEVICE)               # (num_total_instances, res_t, 1, 1, res, res, 1)
params  = torch.cat(params, dim=0).to(DEVICE)           # (num_total_instances, 1)
num_inst, res_t = data.size(0), data.size(1)
data    = data.view(num_inst, res_t, res_v, res_v, 1)
data    = data.type(torch.float)
params  = params.type(torch.float)

print(f"Successfully loaded dataset with {num_inst} instances.")

  0%|          | 0/1 [00:00<?, ?it/s]

Successfully loaded dataset with 1 instances.


### 3-2. Compute the trajectories on the test set

Fast spectral method

In [4]:
torch.cuda.reset_peak_memory_stats(DEVICE)
torch.cuda.synchronize(DEVICE)

fsm = FastSM_Boltzmann_VHS(DIMENSION, res_v, MAX_V, vhs_coeff=VHS_COEFF, vhs_alpha=alpha, dtype=torch.float32, device=DEVICE)
arr_f = data[:, 0].reshape(1, *(1 for _ in range(DIMENSION)), *(res_v for _ in range(DIMENSION)), 1) # (batch, x, v, channel)

inference_time_fsm = perf_counter()
with torch.inference_mode():
    pred_fsm = fsm.solve(min_t, max_t, delta_t, arr_f, one_step_RK4_classic)
inference_time_fsm = perf_counter() - inference_time_fsm
pred_fsm = pred_fsm.reshape(num_inst, res_t, *(res_v for _ in range(DIMENSION)), 1).cpu()

torch.cuda.synchronize(DEVICE)
peak_memory_fsm = torch.cuda.max_memory_allocated(DEVICE) / (1024**2)   # MB
print(f"FSM Peak memory usage: {peak_memory_fsm:.2f}MB.")

100%|██████████| 50/50 [00:00<00:00, 147.05it/s]

FSM Peak memory usage: 347.67MB.


End of file